In [1]:
import os
import zipfile
from pathlib import Path

# 1. Find uploaded ZIP file
zip_files = list(Path(".").glob("*.zip"))

print("ZIP files found:")
for z in zip_files:
    print(z)

# Select first ZIP file
zip_path = zip_files[0]

# 2. Extract dataset
extract_path = "Mushroom_Growth_Dataset"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("\nDataset extracted successfully!")
print("Location:", extract_path)

# 3. Count images
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

for split in ["train", "valid", "test"]:
    image_folder = Path(extract_path) / split / "images"

    if image_folder.exists():
        count = len([
            f for f in image_folder.iterdir()
            if f.suffix.lower() in image_extensions
        ])

        print(f"{split} images: {count}")
    else:
        print(f"{split} folder not found")

ZIP files found:
Mushroom Growth Stages.v2-v1.yolov8.zip
MushroomGrowthStages.v3i.yolov8.zip

Dataset extracted successfully!
Location: Mushroom_Growth_Dataset
train images: 4989
valid images: 717
test images: 717


In [2]:
!find Mushroom_Growth_Dataset -maxdepth 2 -type d

Mushroom_Growth_Dataset
Mushroom_Growth_Dataset/train
Mushroom_Growth_Dataset/train/images
Mushroom_Growth_Dataset/train/labels
Mushroom_Growth_Dataset/test
Mushroom_Growth_Dataset/test/images
Mushroom_Growth_Dataset/test/labels
Mushroom_Growth_Dataset/.ipynb_checkpoints
Mushroom_Growth_Dataset/valid
Mushroom_Growth_Dataset/valid/images
Mushroom_Growth_Dataset/valid/labels


In [3]:
with open("Mushroom_Growth_Dataset/data.yaml", "r") as f:
    print(f.read())

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 3
names: ['Harvest', 'Intermediate', 'Juvenile']

roboflow:
  workspace: praveens-workspace-8qjfs
  project: mushroom-growth-stages-2vmfq-hzkry
  version: 2
  license: CC BY 4.0
  url: https://universe.roboflow.com/praveens-workspace-8qjfs/mushroom-growth-stages-2vmfq-hzkry/dataset/2


In [4]:
yaml_content = """train: train/images
val: valid/images
test: test/images

nc: 3
names: ['Harvest', 'Maturing', 'Juvenile']
"""

with open("Mushroom_Growth_Dataset/data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml updated successfully!")

data.yaml updated successfully!


In [5]:
with open("Mushroom_Growth_Dataset/data.yaml", "r") as f:
    print(f.read())

train: train/images
val: valid/images
test: test/images

nc: 3
names: ['Harvest', 'Maturing', 'Juvenile']



In [7]:
from pathlib import Path
from collections import Counter

class_names = {
    0: "Harvest",
    1: "Maturing",
    2: "Juvenile"
}

for split in ["train", "valid", "test"]:
    
    label_folder = Path("Mushroom_Growth_Dataset") / split / "labels"
    
    counts = Counter()

    for label_file in label_folder.glob("*.txt"):
        with open(label_file, "r") as f:
            for line in f:
                if line.strip():
                    class_id = int(line.split()[0])
                    counts[class_id] += 1

    print(f"\n{split.upper()}")
    
    for class_id, class_name in class_names.items():
        print(f"{class_name}: {counts[class_id]}")


TRAIN
Harvest: 2697
Maturing: 2208
Juvenile: 1504

VALID
Harvest: 445
Maturing: 301
Juvenile: 169

TEST
Harvest: 374
Maturing: 371
Juvenile: 186


In [8]:
import yaml

with open("Mushroom_Growth_Dataset/data.yaml", "r") as f:
    data = yaml.safe_load(f)

print(data["names"])

['Harvest', 'Maturing', 'Juvenile']


In [9]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.9.1+cu128
CUDA: True
GPU: Tesla V100-PCIE-32GB


In [10]:
from ultralytics import YOLO

In [2]:
!nvidia-smi

Tue Aug 25 14:06:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.42.06              Driver Version: 555.42.06      CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-PCIE-32GB           Off |   00000000:3B:00.0 Off |                    0 |
| N/A   47C    P0            161W /  250W |    8077MiB /  32768MiB |     54%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!ps -fp 441524
!ps -fp 443331

UID          PID    PPID  C STIME TTY          TIME CMD
UID          PID    PPID  C STIME TTY          TIME CMD
anishkr+  443331  441498 99 13:47 pts/1    00:34:47 python train.py --backbone e


In [4]:
!kill 441524
!kill 443331

/bin/bash: line 1: kill: (441524) - No such process


In [5]:
!nvidia-smi

Tue Aug 25 14:08:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.42.06              Driver Version: 555.42.06      CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-PCIE-32GB           Off |   00000000:3B:00.0 Off |                    0 |
| N/A   41C    P0             49W /  250W |     911MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")

0.46387767791748047 GB allocated
0.470703125 GB reserved


In [7]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

results = model.train(
    data="Mushroom_Growth_Dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=8,
    patience=20,
    project="Mushroom_Project",
    name="Growth_Stages_YOLOv8m"
)

New https://pypi.org/project/ultralytics/8.4.128 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.17 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (Tesla V100-PCIE-32GB, 32494MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Mushroom_Growth_Dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=

In [1]:
from pathlib import Path

best_models = list(Path(".").rglob("best.pt"))

for model_path in best_models:
    print(model_path)

runs/detect/Mushroom_Project/Growth_Stages_YOLOv8m3/weights/best.pt
runs/detect/Mushroom_Project/Growth_Stages/weights/best.pt
runs/detect/Mushroom_Project/Growth_Stages2/weights/best.pt


In [4]:
from ultralytics import YOLO

model = YOLO("runs/detect/Mushroom_Project/Growth_Stages_YOLOv8m3/weights/best.pt")

print("Model loaded successfully")

Model loaded successfully


In [5]:
metrics = model.val(
    data="Mushroom_Growth_Dataset/data.yaml",
    split="test",
    imgsz=640,
    batch=8,
    device=0,
    plots=True
)

print("\nTest Results")
print(metrics.results_dict)

Ultralytics 8.4.17 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (Tesla V100-PCIE-32GB, 32494MiB)
Model summary (fused): 93 layers, 25,841,497 parameters, 0 gradients
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 36.4±65.5 MB/s, size: 60.4 KB)
val: Scanning /home/anishkrishnas.23cse/23ECR161/Mushroom_Growth_Dataset/test/labels... 717 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 717/717 833.3it/s 0.9s0.0s
val: New cache created: /home/anishkrishnas.23cse/23ECR161/Mushroom_Growth_Dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 90/90 8.9it/s 10.1s<0.1s
                   all        717        931       0.83      0.815      0.904      0.753
               Harvest        325        374      0.846      0.896      0.928      0.798
              Maturing        324        371      0.842      0.854      0.929      0.788
              Juvenile        157        186      0.803      0.694      0.856      0.

In [6]:
metrics = model.val(
    data="Mushroom_Growth_Dataset/data.yaml",
    split="test",
    imgsz=640,
    device=0,
    plots=True
)

print("Precision :", metrics.box.mp)
print("Recall    :", metrics.box.mr)
print("mAP50     :", metrics.box.map50)
print("mAP50-95  :", metrics.box.map)

Ultralytics 8.4.17 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (Tesla V100-PCIE-32GB, 32494MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1508.5±396.8 MB/s, size: 55.1 KB)
val: Scanning /home/anishkrishnas.23cse/23ECR161/Mushroom_Growth_Dataset/test/labels.cache... 717 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 717/717 94.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 6.0it/s 7.5s0.2s
                   all        717        931       0.83      0.815      0.904      0.753
               Harvest        325        374      0.846      0.896      0.928      0.798
              Maturing        324        371      0.842      0.854      0.929      0.788
              Juvenile        157        186      0.803      0.694      0.856      0.674
Speed: 1.0ms preprocess, 6.7ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /home/anishkrishnas.23cse/23ECR161/runs/detect/val4
Precision : 

In [8]:
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt

model = YOLO(
    "runs/detect/Mushroom_Project/Growth_Stages_YOLOv8m3/weights/best.pt"
)

test_images = list(
    Path("Mushroom_Growth_Dataset/test/images").glob("*")
)

results = model.predict(
    source=str(test_images[0]),
    imgsz=640,
    conf=0.25
)

img = results[0].plot()

plt.figure(figsize=(10, 8))
plt.imshow(img[..., ::-1])
plt.axis("off")
plt.show()


image 1/1 /home/anishkrishnas.23cse/23ECR161/Mushroom_Growth_Dataset/test/images/IMG_7884_MOV-0042_jpg.rf.992a195d66de00f86ffb7a9d187126ac.jpg: 640x640 1 Maturing, 10.7ms
Speed: 1.3ms preprocess, 10.7ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 640)


<Figure size 1000x800 with 1 Axes>

In [10]:
from ultralytics import YOLO

# Load the trained model
model = YOLO("runs/detect/Mushroom_Project/Growth_Stages_YOLOv8m3/weights/best.pt")   # Change path if needed

# Predict on all test images
results = model.predict(
    source="Mushroom_Growth_Dataset/test/images",   # Path to your test images
    imgsz=640,
    conf=0.25,
    save=True
)

print("Prediction completed!")


image 1/717 /home/anishkrishnas.23cse/23ECR161/Mushroom_Growth_Dataset/test/images/IMG_1578_mov-0386_jpg.rf.0f974bfb8a184136f13ce0e66151e8fe.jpg: 640x640 1 Harvest, 10.7ms
image 2/717 /home/anishkrishnas.23cse/23ECR161/Mushroom_Growth_Dataset/test/images/IMG_1578_mov-0438_jpg.rf.369da985deaad219639ff1b9668ea7a6.jpg: 640x640 1 Maturing, 10.6ms
image 3/717 /home/anishkrishnas.23cse/23ECR161/Mushroom_Growth_Dataset/test/images/IMG_1578_mov-0461_jpg.rf.00fc2bab9270c77c1c3ee0f16be93ea0.jpg: 640x640 1 Maturing, 10.6ms
image 4/717 /home/anishkrishnas.23cse/23ECR161/Mushroom_Growth_Dataset/test/images/IMG_1578_mov-0487_jpg.rf.885428144d97d3fe33d140df0779fdd8.jpg: 640x640 1 Harvest, 10.6ms
image 5/717 /home/anishkrishnas.23cse/23ECR161/Mushroom_Growth_Dataset/test/images/IMG_1578_mov-0529_jpg.rf.fece3973178571bf848d00833ec10459.jpg: 640x640 1 Maturing, 1 Juvenile, 10.7ms
image 6/717 /home/anishkrishnas.23cse/23ECR161/Mushroom_Growth_Dataset/test/images/IMG_1578_mov-0535_jpg.rf.4fe5bdc4e5140800